# TALC Classification Analysis: Confidence Intervals, Chi-Square Test, and Location Quotient

This notebook reproduces the statistical analyses reported in the manuscript that underlie Table 2 (sentiment distribution with 95% confidence intervals), the chi-square test of sentiment differences across TALC stages, Table 4 (Location Quotient values), and the pre-pandemic robustness check. It uses `All_Reviews_Dataset.xlsx` (published in this repository) and `Problem_Category_Counts.csv` (the destination-by-category counts underlying Table 3, also published in this repository).

## 1. Import Libraries

In [ ]:
import math
import pandas as pd
from scipy.stats import chi2_contingency

## 2. Load the Dataset

In [ ]:
df = pd.read_excel("All_Reviews_Dataset.xlsx")
df.head()

## 3. Wilson Score Confidence Intervals (Table 2)

95% confidence intervals for the sentiment proportion at each destination, using the Wilson score method rather than the normal approximation.

In [ ]:
def wilson_ci(count, n, z=1.96):
    p = count / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z * math.sqrt((p * (1 - p)) / n + z**2 / (4 * n**2))) / denom
    return p * 100, (center - margin) * 100, (center + margin) * 100

rows = []
for dest, g in df.groupby("Destination"):
    n = len(g)
    for cls in ["Positive", "Neutral", "Negative"]:
        count = (g["Sentiment_Category"] == cls).sum()
        pct, lo, hi = wilson_ci(count, n)
        rows.append({"Destination": dest, "Class": cls, "Percent": round(pct, 2),
                      "CI_low": round(lo, 2), "CI_high": round(hi, 2), "n": n})

ci_table = pd.DataFrame(rows)
ci_table

## 4. Chi-Square Test of Independence and Cramer's V

Tests whether sentiment distributions differ across the three TALC stage groupings (mature, developing, involvement/emerging), as assigned in Table 4.

In [ ]:
talc_stage = {
    "Pattaya Beach": "Mature",
    "Koh Lan": "Mature",
    "Bangsaen Beach": "Developing",
    "Koh Samet": "Developing",
    "Koh Chang": "Developing",
    "Chao Lao Beach": "Involvement/Emerging",
    "Koh Kood": "Involvement/Emerging",
}
df["TALC_Stage"] = df["Destination"].map(talc_stage)

contingency = pd.crosstab(df["TALC_Stage"], df["Sentiment_Category"])
contingency = contingency[["Positive", "Neutral", "Negative"]]
print(contingency)

chi2, p, dof, expected = chi2_contingency(contingency)
n = contingency.values.sum()
k = min(contingency.shape) - 1
cramers_v = (chi2 / (n * k)) ** 0.5
print(f"\nchi2 = {chi2:.2f}, df = {dof}, p = {p:.3e}, Cramer's V = {cramers_v:.3f}")

## 5. Location Quotient (Table 4)

LQ is defined as a destination's percentage share of a given problem category, divided by that category's overall percentage share across all seven destinations. Values above 1 indicate over-representation relative to the full sample.

In [ ]:
issues = pd.read_csv("Problem_Category_Counts.csv", index_col="Destination")
categories = [c for c in issues.columns if c != "Total"]

overall_share = issues[categories].sum() / issues[categories].sum().sum()

lq_table = pd.DataFrame(index=issues.index, columns=categories, dtype=float)
for dest in issues.index:
    dest_share = issues.loc[dest, categories] / issues.loc[dest, "Total"]
    lq_table.loc[dest] = (dest_share / overall_share).round(2)

lq_table["Predominant_category"] = lq_table[categories].idxmax(axis=1)
lq_table["Predominant_LQ"] = lq_table[categories].max(axis=1)
lq_table

## 6. Pre-Pandemic Robustness Check

Negative sentiment proportions recalculated using only 2014-2019 reviews, compared against the full 2014-2023 study period.

In [ ]:
df["Year"] = pd.to_datetime(df["Review_Date"]).dt.year

rows = []
for dest, g in df.groupby("Destination"):
    pre = g[g["Year"] <= 2019]
    pre_pct = (pre["Sentiment_Category"] == "Negative").mean() * 100
    full_pct = (g["Sentiment_Category"] == "Negative").mean() * 100
    rows.append({"Destination": dest, "Pre_pandemic_2014_2019": round(pre_pct, 2),
                  "Full_period_2014_2023": round(full_pct, 2),
                  "Difference_pp": round(abs(pre_pct - full_pct), 2)})

pd.DataFrame(rows)

## 7. Expected Output

- Chi-square test: chi2 = 399.45, df = 4, p < .001, Cramer's V = 0.123
- Location Quotient (Table 4): matches reported values for all destinations and categories, within 0.01 for two cells (Bangsaen Beach / Pollution and littering; Koh Kood / Accessibility challenges), attributable to intermediate rounding in the original calculation.
- Robustness check: all destinations differ by 0.83 percentage points or less between the pre-pandemic and full-period negative sentiment proportions, except Chao Lao Beach (4.55% vs 8.33%, a difference of 3.79 percentage points), consistent with its small sample size.